In [3]:
"""
Notes as well as a uysage guide made from AI\
This file takes the raw peptide/HLA CSV and turns it into data the PyTorch model can actually train on.

The basic biology idea is simple:

    peptide = short amino acid chain
    HLA = molecule that holds the peptide
    cancer label = whether the peptide is cancer-related or healthy

The model gets 3 views of each row:

    1. bio
       Small chemistry facts about the peptide, like weight, charge, hydrophobicity,
       stability, etc.

    2. pep
       The actual peptide amino acid letters changed into BLOSUM62 numbers.
       BLOSUM is used because amino acids are not random letters. Some amino acids
       behave more similarly than others, so BLOSUM gives the model biology-based
       numbers instead of fake number IDs.

    3. mhc
       The HLA binding groove sequence changed into BLOSUM62 numbers.
       We do not use the whole HLA protein because the binding groove is the main
       part that holds the peptide.

The target label is:

    is_cancer
        1 = cancer
        0 = healthy


The file works in this order:

    get_hla_library()
        Stores the HLA alleles the model knows.
        Each allele has its full HLA protein sequence.

    encode_blosum(seq)
        Turns amino acid letters into BLOSUM62 number rows.
        A peptide like "SLYNTVATL" becomes a matrix with one biology-number row
        per amino acid.

    hla_pseudosequence(allele, hla_library)
        Takes one HLA allele and cuts out only the important binding-groove positions.
        This makes the HLA input smaller and more focused on peptide binding.

    make_pseudo_map(hla_library)
        Builds a quick lookup table:
            HLA allele name -> HLA groove pseudosequence
        This lets each dataset row quickly find its HLA sequence.

    load_and_filter_data(path, hla_library)
        Loads the CSV.
        Drops columns that can leak the answer or act like IDs.
        Keeps only rows that have:
            is_cancer
            Peptide Sequence
            Best HLA Allele
        Also removes rows with HLA alleles not in the HLA library.

    make_splits(df)
        Splits the cleaned data into train, validation, and test.
        Stratify keeps cancer/healthy balance similar in each split.

    get_numeric_bio_candidates(train_df)
        Finds numeric peptide feature columns that are allowed to be used.
        It skips the label, peptide sequence, and HLA allele because those are handled separately.

    clean_numeric_frame(frame, columns, fill_values=None)
        Fixes missing or infinity values in numeric bio features.
        Training data creates the fill values.
        Validation/test reuse training fill values so we do not peek at them.

    select_top_bio_features(train_df)
        Uses Random Forest only to rank the peptide chemistry features.
        Keeps the top 8 strongest bio features.
        Random Forest is not the final model. It is just choosing useful features.

    make_bio_scaler(train_df, top_features)
        Scales the top bio features so big-number columns do not overpower small-number columns.

    PeptideMHCDataset
        Converts one dataframe row into PyTorch tensors:
            bio       = scaled top 8 chemistry features
            pep       = padded peptide BLOSUM matrix
            pep_mask  = tells attention what is real amino acid vs padding
            mhc       = HLA groove BLOSUM matrix
            label     = cancer/healthy

    make_datasets()
        Turns train_df, val_df, and test_df into PyTorch Dataset objects.

    make_loaders()
        Turns the Dataset objects into DataLoaders.
        These make batches for training.

    check_batch()
        Grabs one batch and checks the shapes.

    build_preprocessing_pipeline()
        Runs everything above in the correct order and returns all useful objects.


After this file runs, the main things used for training are:

    train_loader
    val_loader
    test_loader

A batch should look like:

    bio       -> (batch_size, 8)
    pep       -> (batch_size, 11, 20)
    pep_mask  -> (batch_size, 11)
    mhc       -> (batch_size, 34, 20)
    label     -> (batch_size,)

So the whole point is:

    raw CSV
        -> clean rows
        -> remove leaky columns
        -> split data
        -> choose useful peptide chemistry features
        -> encode peptide letters with BLOSUM
        -> encode HLA binding groove with BLOSUM
        -> build PyTorch batches
        -> ready for the model
"""

'\nNotes as well as a uysage guide made from AIThis file takes the raw peptide/HLA CSV and turns it into data the PyTorch model can actually train on.\n\nThe basic biology idea is simple:\n\n    peptide = short amino acid chain\n    HLA = molecule that holds the peptide\n    cancer label = whether the peptide is cancer-related or healthy\n\nThe model gets 3 views of each row:\n\n    1. bio\n       Small chemistry facts about the peptide, like weight, charge, hydrophobicity,\n       stability, etc.\n\n    2. pep\n       The actual peptide amino acid letters changed into BLOSUM62 numbers.\n       BLOSUM is used because amino acids are not random letters. Some amino acids\n       behave more similarly than others, so BLOSUM gives the model biology-based\n       numbers instead of fake number IDs.\n\n    3. mhc\n       The HLA binding groove sequence changed into BLOSUM62 numbers.\n       We do not use the whole HLA protein because the binding groove is the main\n       part that holds the

In [ ]:
a

Using device: cpu
Device: cpu
Rows: 43,678
Alleles used: 27
Train: 30,573 | Val: 6,553 | Test: 6,552
Top 8 features: ['molecular_weight', 'tcr_contact_hydro_mean', 'boman_index', 'hydrophobicity_GRAVY', 'instability_index', 'charge_pH_7', 'isoelectric_point', 'aliphatic_index']
Batch shapes: {'bio': (128, 8), 'pep': (128, 11, 20), 'pep_mask': (128, 11), 'mhc': (128, 34, 20), 'label': (128,)}


In [7]:
import copy
from contextlib import nullcontext

import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


CFG = {
    "d_model": 96,
    "n_heads": 4,
    "n_layers_pep": 3,
    "n_layers_mhc": 2,
    "dropout": 0.25,

    "lr": 7e-4,
    "weight_decay": 1e-3,
    "epochs": 40,
    "patience": 8,
    "grad_clip": 1.0,

    "label_smoothing": 0.03,
    "use_class_weights": False,

    "score_metric": "val_acc",
    "checkpoint_name": "better_threestage_classI_best.pt",
}


def seed_everything(seed=RANDOM_STATE):
    torch.manual_seed(seed)
    np.random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def to_device(batch):
    return (
        batch["bio"].to(device),
        batch["pep"].to(device),
        batch["pep_mask"].to(device),
        batch["mhc"].to(device),
        batch["label"].to(device),
    )


def get_class_weights(train_df):
    counts = np.bincount(train_df["is_cancer"].astype(int).values)
    weights = counts.sum() / (2.0 * counts)
    return torch.tensor(weights, dtype=torch.float32).to(device)


class BetterThreeStageModel(nn.Module):
    def __init__(
        self,
        n_bio=N_TOP_BIO,
        d_model=96,
        n_heads=4,
        n_layers_pep=3,
        n_layers_mhc=2,
        max_pep=MAX_PEP_LEN,
        dropout=0.25,
    ):
        super().__init__()

        self.bio_mlp = nn.Sequential(
            nn.Linear(n_bio, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )

        self.pep_proj = nn.Linear(20, d_model)
        self.pep_pos = nn.Embedding(max_pep, d_model)

        pep_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.pep_encoder = nn.TransformerEncoder(
            pep_layer,
            num_layers=n_layers_pep,
            enable_nested_tensor=False,
        )

        self.mhc_proj = nn.Linear(20, d_model)
        self.mhc_pos = nn.Embedding(PSEUDO_LEN, d_model)

        mhc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.mhc_encoder = nn.TransformerEncoder(
            mhc_layer,
            num_layers=n_layers_mhc,
            enable_nested_tensor=False,
        )

        self.pep_to_mhc = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.mhc_to_pep = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.pep_cross_norm = nn.LayerNorm(d_model)
        self.mhc_cross_norm = nn.LayerNorm(d_model)

        self.fusion_attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.fusion_norm = nn.LayerNorm(d_model)

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model * 5),
            nn.Linear(d_model * 5, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(d_model, 2),
        )

    def masked_mean(self, x, pad_mask):
        keep = (~pad_mask).unsqueeze(-1).float()
        return (x * keep).sum(dim=1) / keep.sum(dim=1).clamp(min=1.0)

    def forward(self, bio, pep, pep_mask, mhc):
        B = bio.size(0)

        bio_tok = self.bio_mlp(bio)

        pep_pos = torch.arange(pep.size(1), device=pep.device).unsqueeze(0)
        pep_x = self.pep_proj(pep) + self.pep_pos(pep_pos)
        pep_x = self.pep_encoder(pep_x, src_key_padding_mask=pep_mask)
        pep_tok = self.masked_mean(pep_x, pep_mask)

        mhc_pos = torch.arange(mhc.size(1), device=mhc.device).unsqueeze(0)
        mhc_x = self.mhc_proj(mhc) + self.mhc_pos(mhc_pos)
        mhc_x = self.mhc_encoder(mhc_x)
        mhc_tok = mhc_x.mean(dim=1)

        pep_cross, _ = self.pep_to_mhc(
            query=pep_x,
            key=mhc_x,
            value=mhc_x,
        )

        pep_cross = self.pep_cross_norm(pep_x + pep_cross)
        pep_mhc_tok = self.masked_mean(pep_cross, pep_mask)

        mhc_cross, _ = self.mhc_to_pep(
            query=mhc_x,
            key=pep_x,
            value=pep_x,
            key_padding_mask=pep_mask,
        )

        mhc_cross = self.mhc_cross_norm(mhc_x + mhc_cross)
        mhc_pep_tok = mhc_cross.mean(dim=1)

        tokens = torch.stack(
            [bio_tok, pep_tok, mhc_tok, pep_mhc_tok, mhc_pep_tok],
            dim=1,
        )

        fused, _ = self.fusion_attn(tokens, tokens, tokens)
        fused = self.fusion_norm(tokens + fused)

        logits = self.classifier(fused.reshape(B, -1))
        return logits

In [8]:
def build_model(cfg):
    model = BetterThreeStageModel(
        d_model=cfg["d_model"],
        n_heads=cfg["n_heads"],
        n_layers_pep=cfg["n_layers_pep"],
        n_layers_mhc=cfg["n_layers_mhc"],
        dropout=cfg["dropout"],
    ).to(device)

    return model


def build_loss(cfg):
    if cfg["use_class_weights"]:
        weights = get_class_weights(train_df)
    else:
        weights = None

    return nn.CrossEntropyLoss(
        weight=weights,
        label_smoothing=cfg["label_smoothing"],
    )


def build_optimizer(model, cfg):
    return torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
    )


def build_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
    )


def dry_run_model(model, loss_fn):
    model.train()

    batch = next(iter(train_loader))
    bio, pep, pep_mask, mhc, label = to_device(batch)

    logits = model(bio, pep, pep_mask, mhc)

    assert logits.shape == (bio.size(0), 2), f"Bad output shape: {logits.shape}"
    assert not torch.isnan(logits).any(), "NaNs found in logits"

    loss = loss_fn(logits, label)
    loss.backward()

    grad_norm = sum(
        p.grad.norm().item()
        for p in model.parameters()
        if p.grad is not None
    )

    print("Dry run passed")
    print("Output shape:", tuple(logits.shape))
    print("Loss:", round(loss.item(), 4))
    print("Grad norm:", round(grad_norm, 4))

    model.zero_grad()


def run_epoch(model, loader, loss_fn, optimizer=None, scaler=None, cfg=None):
    is_train = optimizer is not None

    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_true = []
    all_pred = []
    all_prob = []

    use_amp = device.type == "cuda"

    with torch.set_grad_enabled(is_train):
        for batch in loader:
            bio, pep, pep_mask, mhc, label = to_device(batch)

            amp_context = (
                torch.autocast(device_type="cuda", dtype=torch.float16)
                if use_amp
                else nullcontext()
            )

            with amp_context:
                logits = model(bio, pep, pep_mask, mhc)
                loss = loss_fn(logits, label)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

                if use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                    optimizer.step()

            total_loss += loss.item() * label.size(0)

            prob = torch.softmax(logits.detach(), dim=1)[:, 1]
            pred = logits.detach().argmax(dim=1)

            all_true.extend(label.cpu().tolist())
            all_pred.extend(pred.cpu().tolist())
            all_prob.extend(prob.cpu().tolist())

    avg_loss = total_loss / len(all_true)
    acc = accuracy_score(all_true, all_pred)
    bal_acc = balanced_accuracy_score(all_true, all_pred)

    try:
        auc = roc_auc_score(all_true, all_prob)
    except ValueError:
        auc = 0.0

    return {
        "loss": avg_loss,
        "acc": acc,
        "bal_acc": bal_acc,
        "auc": auc,
        "true": all_true,
        "pred": all_pred,
        "prob": all_prob,
    }


def get_score(metrics, metric_name):
    if metric_name == "val_acc":
        return metrics["acc"]

    if metric_name == "val_bal_acc":
        return metrics["bal_acc"]

    if metric_name == "val_auc":
        return metrics["auc"]

    raise ValueError("score_metric must be: val_acc, val_bal_acc, or val_auc")


def train_model(cfg):
    seed_everything()

    model = build_model(cfg)
    loss_fn = build_loss(cfg)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer)

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    ckpt_path = os.path.join(CACHE_DIR, cfg["checkpoint_name"])

    print(model)
    print("Trainable params:", count_params(model))

    dry_run_model(model, loss_fn)

    best_score = -1
    best_state = None
    wait = 0

    hist = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "train_bal_acc": [],
        "val_bal_acc": [],
        "train_auc": [],
        "val_auc": [],
    }

    for epoch in range(1, cfg["epochs"] + 1):
        train_metrics = run_epoch(
            model,
            train_loader,
            loss_fn,
            optimizer=optimizer,
            scaler=scaler,
            cfg=cfg,
        )

        val_metrics = run_epoch(
            model,
            val_loader,
            loss_fn,
            cfg=cfg,
        )

        hist["train_loss"].append(train_metrics["loss"])
        hist["val_loss"].append(val_metrics["loss"])
        hist["train_acc"].append(train_metrics["acc"])
        hist["val_acc"].append(val_metrics["acc"])
        hist["train_bal_acc"].append(train_metrics["bal_acc"])
        hist["val_bal_acc"].append(val_metrics["bal_acc"])
        hist["train_auc"].append(train_metrics["auc"])
        hist["val_auc"].append(val_metrics["auc"])

        score = get_score(val_metrics, cfg["score_metric"])
        scheduler.step(score)

        print(
            f"Epoch {epoch:02d} | "
            f"train loss {train_metrics['loss']:.4f} "
            f"acc {train_metrics['acc']:.4f} "
            f"bal {train_metrics['bal_acc']:.4f} "
            f"auc {train_metrics['auc']:.4f} | "
            f"val loss {val_metrics['loss']:.4f} "
            f"acc {val_metrics['acc']:.4f} "
            f"bal {val_metrics['bal_acc']:.4f} "
            f"auc {val_metrics['auc']:.4f}"
        )

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, ckpt_path)
            wait = 0
        else:
            wait += 1

            if wait >= cfg["patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

    print(f"\nBest {cfg['score_metric']}: {best_score:.4f}")
    print(f"Saved to: {ckpt_path}")

    model.load_state_dict(torch.load(ckpt_path, map_location=device))

    return model, hist, ckpt_path

In [12]:
def collect_outputs(model, loader):
    model.eval()

    all_true = []
    all_prob = []

    with torch.no_grad():
        for batch in loader:
            bio, pep, pep_mask, mhc, label = to_device(batch)

            logits = model(bio, pep, pep_mask, mhc)
            prob = torch.softmax(logits, dim=1)[:, 1]

            all_true.extend(label.cpu().tolist())
            all_prob.extend(prob.cpu().tolist())

    return np.array(all_true), np.array(all_prob)


def find_best_threshold(y_true, y_prob, metric="accuracy"):
    best_t = 0.5
    best_score = -1

    thresholds = np.linspace(0.05, 0.95, 181)

    for t in thresholds:
        pred = (y_prob >= t).astype(int)

        if metric == "accuracy":
            score = accuracy_score(y_true, pred)
        elif metric == "balanced_accuracy":
            score = balanced_accuracy_score(y_true, pred)
        else:
            raise ValueError("metric must be accuracy or balanced_accuracy")

        if score > best_score:
            best_score = score
            best_t = t

    return best_t, best_score


def evaluate_model(model, loader, threshold=0.5, title="TEST SET"):
    y_true, y_prob = collect_outputs(model, loader)
    y_pred = (y_prob >= threshold).astype(int)

    print(f"===== {title} =====")
    print(f"Threshold:         {threshold:.3f}")
    print(f"Accuracy:          {accuracy_score(y_true, y_pred):.4f}")
    print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")
    print(f"ROC AUC:           {roc_auc_score(y_true, y_prob):.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["healthy", "cancer"],
            digits=4,
        )
    )

    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Blues")

    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=12)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["healthy", "cancer"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["healthy", "cancer"])
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title)

    plt.tight_layout()
    plt.show()

    return {
        "true": y_true,
        "prob": y_prob,
        "pred": y_pred,
    }


def plot_history(hist):
    plt.figure(figsize=(7, 4))
    plt.plot(hist["train_acc"], label="train acc")
    plt.plot(hist["val_acc"], label="val acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title("Accuracy curve")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(hist["train_auc"], label="train auc")
    plt.plot(hist["val_auc"], label="val auc")
    plt.xlabel("epoch")
    plt.ylabel("ROC AUC")
    plt.title("AUC curve")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [13]:
model, hist, ckpt_path = train_model(CFG)

plot_history(hist)

val_true, val_prob = collect_outputs(model, val_loader)

best_threshold, best_val_acc = find_best_threshold(
    val_true,
    val_prob,
    metric="accuracy",
)

print(f"Best validation threshold: {best_threshold:.3f}")
print(f"Best validation accuracy with threshold: {best_val_acc:.4f}")

evaluate_model(
    model,
    test_loader,
    threshold=0.5,
    title="TEST SET DEFAULT THRESHOLD",
)

evaluate_model(
    model,
    test_loader,
    threshold=best_threshold,
    title="TEST SET VAL-TUNED THRESHOLD",
)

C:\Users\Waheg\AppData\Local\Temp\ipykernel_36148\1230672098.py:160: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


BetterThreeStageModel(
  (bio_mlp): Sequential(
    (0): Linear(in_features=8, out_features=96, bias=True)
    (1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.25, inplace=False)
    (4): Linear(in_features=96, out_features=96, bias=True)
    (5): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    (6): GELU(approximate='none')
  )
  (pep_proj): Linear(in_features=20, out_features=96, bias=True)
  (pep_pos): Embedding(11, 96)
  (pep_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=96, out_features=96, bias=True)
        )
        (linear1): Linear(in_features=96, out_features=384, bias=True)
        (dropout): Dropout(p=0.25, inplace=False)
        (linear2): Linear(in_features=384, out_features=96, bias=True)
        (norm1): LayerNorm((96,), eps=1e-05, elementwis

KeyboardInterrupt: 

In [14]:
CFG["d_model"] = 96
CFG["dropout"] = 0.25
CFG["lr"] = 7e-4
CFG["weight_decay"] = 1e-3
CFG["score_metric"] = "val_acc"
CFG["use_class_weights"] = False

In [15]:
CFG.update({
    "d_model": 160,
    "n_heads": 4,
    "n_layers_pep": 4,
    "n_layers_mhc": 2,
    "dropout": 0.15,

    "lr": 7e-4,
    "weight_decay": 1e-4,
    "epochs": 60,
    "patience": 10,

    "label_smoothing": 0.0,
    "use_class_weights": False,

    "score_metric": "val_acc",
    "checkpoint_name": "better_threestage_bigger.pt",
})

In [16]:
configs = []

cfg1 = CFG.copy()
cfg1.update({
    "d_model": 128,
    "dropout": 0.12,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "label_smoothing": 0.0,
    "score_metric": "val_acc",
    "checkpoint_name": "run1_128_lowdrop.pt",
})
configs.append(cfg1)

cfg2 = CFG.copy()
cfg2.update({
    "d_model": 160,
    "dropout": 0.15,
    "lr": 7e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.0,
    "score_metric": "val_acc",
    "checkpoint_name": "run2_160.pt",
})
configs.append(cfg2)

cfg3 = CFG.copy()
cfg3.update({
    "d_model": 128,
    "dropout": 0.20,
    "lr": 5e-4,
    "weight_decay": 5e-5,
    "label_smoothing": 0.0,
    "score_metric": "val_acc",
    "checkpoint_name": "run3_128_slower.pt",
})
configs.append(cfg3)

In [17]:
trained_models = []

for i, cfg in enumerate(configs, 1):
    print(f"\n===== TRAINING MODEL {i} =====")
    model, hist, ckpt_path = train_model(cfg)
    trained_models.append(model)


===== TRAINING MODEL 1 =====
BetterThreeStageModel(
  (bio_mlp): Sequential(
    (0): Linear(in_features=8, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.12, inplace=False)
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (6): GELU(approximate='none')
  )
  (pep_proj): Linear(in_features=20, out_features=128, bias=True)
  (pep_pos): Embedding(11, 128)
  (pep_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.12, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1

C:\Users\Waheg\AppData\Local\Temp\ipykernel_36148\1230672098.py:160: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


Dry run passed
Output shape: (128, 2)
Loss: 0.7001
Grad norm: 1.4229
Epoch 01 | train loss 0.4565 acc 0.7668 bal 0.7671 auc 0.8615 | val loss 0.3652 acc 0.8215 bal 0.8218 auc 0.9162
Epoch 02 | train loss 0.3727 acc 0.8186 bal 0.8186 auc 0.9113 | val loss 0.3683 acc 0.8221 bal 0.8216 auc 0.9166
Epoch 03 | train loss 0.3665 acc 0.8228 bal 0.8228 auc 0.9141 | val loss 0.3616 acc 0.8205 bal 0.8200 auc 0.9182
Epoch 04 | train loss 0.3666 acc 0.8229 bal 0.8231 auc 0.9141 | val loss 0.3636 acc 0.8218 bal 0.8213 auc 0.9152
Epoch 05 | train loss 0.3721 acc 0.8167 bal 0.8168 auc 0.9110 | val loss 0.3580 acc 0.8219 bal 0.8215 auc 0.9179
Epoch 06 | train loss 0.3593 acc 0.8218 bal 0.8219 auc 0.9167 | val loss 0.3563 acc 0.8251 bal 0.8251 auc 0.9190


KeyboardInterrupt: 